# 01 · Inspect a KempnerForge Model

- **Objectives**: build a Transformer from `ModelConfig`, inspect its module tree and parameter count, and run a forward pass end-to-end.
- **Requirements**: 1 GPU (CPU also works for this tiny config).
- **Runtime**: ~5 seconds.

## Setup

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Build a tiny model

`ModelConfig` captures every architecture hyperparameter as a typed dataclass. Changing a single field (e.g., `num_experts`) swaps dense MLPs for MoE blocks with no other code changes.

In [ ]:
from kempnerforge.config.schema import ModelConfig
from kempnerforge.model.transformer import Transformer

config = ModelConfig(
    dim=256,
    n_layers=4,
    n_heads=4,
    n_kv_heads=4,
    vocab_size=4096,
    max_seq_len=512,
)

model = Transformer(config).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,} ({n_params / 1e6:.2f}M)")
print(f"Estimated (from config): {config.num_params_estimate:,}")

## Module tree

The model is a standard pre-norm Llama-style decoder: token embedding → N transformer blocks → final norm → output head. Layers live in a `ModuleDict` keyed by integer index — this preserves fully-qualified names (FQNs) for DCP checkpointing.

In [ ]:
print(model)

## Parameter count by top-level component

In [ ]:
by_component = {}
for name, p in model.named_parameters():
    top = name.split(".", 1)[0]
    by_component[top] = by_component.get(top, 0) + p.numel()

for name, count in sorted(by_component.items(), key=lambda kv: -kv[1]):
    pct = 100 * count / n_params
    print(f"  {name:<20} {count:>12,}  ({pct:4.1f}%)")

## Forward pass

Pass a batch of random token IDs and inspect the logit shape. The model returns `(batch, seq_len, vocab_size)` — raw logits, ready for cross-entropy against next-token targets.

In [ ]:
batch_size = 2
seq_len = 128
tokens = torch.randint(0, config.vocab_size, (batch_size, seq_len), device=device)

with torch.no_grad():
    logits = model(tokens)

print(f"Input tokens:  {tuple(tokens.shape)}  (dtype={tokens.dtype})")
print(f"Output logits: {tuple(logits.shape)}  (dtype={logits.dtype})")

## Swap to MoE with one config change

Setting `num_experts > 0` turns MLP blocks into `MoEMLP` (router + experts). By default (`moe_frequency=1`) every block is MoE; set `moe_frequency=2` for alternating dense/MoE layers. Parameter count grows because we now carry 8 experts per layer instead of one MLP.

In [ ]:
moe_config = ModelConfig(
    dim=256,
    n_layers=4,
    n_heads=4,
    n_kv_heads=4,
    vocab_size=4096,
    max_seq_len=512,
    num_experts=8,
    moe_top_k=2,
)

moe_model = Transformer(moe_config).to(device)
n_moe_params = sum(p.numel() for p in moe_model.parameters())
print(f"Dense model:  {n_params:>12,} params")
print(f"MoE model:    {n_moe_params:>12,} params  ({n_moe_params / n_params:.1f}x)")
print()
print("Block MLPs are now MoEMLP:")
print(type(moe_model.layers['0'].mlp).__name__)

## Summary

- `Transformer(ModelConfig(...))` is the whole model API — no builder pattern, no plugin system.
- The module tree uses `ModuleDict` for layers (so FQNs are stable for distributed checkpointing).
- Switching dense ↔ MoE is a single `num_experts` flag.

Next: see [`02_attention_visualization.ipynb`](02_attention_visualization.ipynb) for how to inspect what the model is *doing* inside those blocks.